In [1]:
import imdlib as imd
from pathlib import Path

In [2]:
raw = Path("../data/raw/imd")
raw.mkdir(parents=True, exist_ok=True)
start, end = 1981, 2023

In [3]:
for var in ["tmax", "tmin", "rain"]:          # small files first
    imd.get_data(var, start, end, fn_format="yearwise", file_dir=str(raw))

Downloading: maxtemp for year 1981
Downloading: maxtemp for year 1982
Downloading: maxtemp for year 1983
Downloading: maxtemp for year 1984
Downloading: maxtemp for year 1985
Downloading: maxtemp for year 1986
Downloading: maxtemp for year 1987
Downloading: maxtemp for year 1988
Downloading: maxtemp for year 1989
Downloading: maxtemp for year 1990
Downloading: maxtemp for year 1991
Downloading: maxtemp for year 1992
Downloading: maxtemp for year 1993
Downloading: maxtemp for year 1994
Downloading: maxtemp for year 1995
Downloading: maxtemp for year 1996
Downloading: maxtemp for year 1997
Downloading: maxtemp for year 1998
Downloading: maxtemp for year 1999
Downloading: maxtemp for year 2000
Downloading: maxtemp for year 2001
Downloading: maxtemp for year 2002
Downloading: maxtemp for year 2003
Downloading: maxtemp for year 2004
Downloading: maxtemp for year 2005
Downloading: maxtemp for year 2006
Downloading: maxtemp for year 2007
Downloading: maxtemp for year 2008
Downloading: maxtemp

In [4]:
for var in ["rain", "tmax", "tmin"]:
    files = sorted((raw / var).glob("*.grd"))
    size = sum(f.stat().st_size for f in files) / 1e6
    print(var, len(files), "files,", files[0].name, "to", files[-1].name, f"{size:.0f} MB")

rain 43 files, 1981.grd to 2023.grd 1094 MB
tmax 43 files, 1981.GRD to 2023.GRD 60 MB
tmin 43 files, 1981.GRD to 2023.GRD 60 MB


In [5]:
import xarray as xr

def load(var):
    da = imd.open_data(var, start, end, "yearwise", str(raw)).get_xarray()[var]
    # thresholds, not ==, because the files store float32 values
    return da.where(da >= 0) if var == "rain" else da.where(da < 99)

In [6]:
rain, tmax, tmin = load("rain"), load("tmax"), load("tmin")

for name, da in [("rain", rain), ("tmax", tmax), ("tmin", tmin)]:
    print(name, da.sizes["time"], "days |",
          float(da.lat.min()), "to", float(da.lat.max()), "°N |",
          float(da.lon.min()), "to", float(da.lon.max()), "°E |",
          str(da.time[0].values)[:10], "to", str(da.time[-1].values)[:10])

rain 15705 days | 6.5 to 38.5 °N | 66.5 to 100.0 °E | 1981-01-01 to 2023-12-31
tmax 15705 days | 7.5 to 37.5 °N | 67.5 to 97.5 °E | 1981-01-01 to 2023-12-31
tmin 15705 days | 7.5 to 37.5 °N | 67.5 to 97.5 °E | 1981-01-01 to 2023-12-31


In [7]:
mask = xr.open_dataarray("../data/processed/marathwada_mask_025.nc").any("region")
box = dict(lat=slice(16.0, 22.0), lon=slice(73.0, 80.0))   # Marathwada plus a margin


In [8]:
r = rain.sel(**box)
m = mask.reindex_like(r, method="nearest", tolerance=0.01).fillna(False).astype(bool)
print(int(m.sum()), "mask cells after alignment (expect 91)")
print(int((r.isnull() & m).sum()), "missing cell-days inside Marathwada (expect 0)")
annual = r.where(m).resample(time="YS").sum(min_count=1).mean(("lat", "lon"))
print(annual.to_series().round(0).to_string())

91 mask cells after alignment (expect 91)
0 missing cell-days inside Marathwada (expect 0)
time
1981-01-01     828.0
1982-01-01     655.0
1983-01-01    1207.0
1984-01-01     564.0
1985-01-01     607.0
1986-01-01     618.0
1987-01-01     801.0
1988-01-01    1246.0
1989-01-01    1062.0
1990-01-01    1211.0
1991-01-01     594.0
1992-01-01     743.0
1993-01-01     791.0
1994-01-01     622.0
1995-01-01     837.0
1996-01-01     856.0
1997-01-01     813.0
1998-01-01    1106.0
1999-01-01     842.0
2000-01-01     735.0
2001-01-01     791.0
2002-01-01     754.0
2003-01-01     703.0
2004-01-01     689.0
2005-01-01     887.0
2006-01-01     897.0
2007-01-01     712.0
2008-01-01     636.0
2009-01-01     707.0
2010-01-01    1030.0
2011-01-01     625.0
2012-01-01     562.0
2013-01-01     962.0
2014-01-01     593.0
2015-01-01     552.0
2016-01-01     986.0
2017-01-01     772.0
2018-01-01     608.0
2019-01-01     869.0
2020-01-01    1014.0
2021-01-01    1180.0
2022-01-01     964.0
2023-01-01     727.0
F

In [9]:
tx, tn = tmax.sel(**box), tmin.sel(**box)
print(int(tx.isnull().sum()), int(tn.isnull().sum()), "missing cell-days (tmax, tmin)")
print(int((tx < tn).sum()), "cell-days with Tmax < Tmin (expect 0)")
clim = xr.Dataset({"tmax": tx, "tmin": tn}).mean(("lat", "lon")).groupby("time.month").mean()
print(clim.to_dataframe().round(1))

0 0 missing cell-days (tmax, tmin)
0 cell-days with Tmax < Tmin (expect 0)
       tmax  tmin
month            
1      29.8  14.4
2      32.5  16.4
3      36.0  19.9
4      38.8  23.5
5      39.6  25.6
6      34.9  24.4
7      30.7  23.1
8      29.8  22.5
9      31.1  22.3
10     32.3  20.5
11     31.1  17.3
12     29.7  14.6


In [10]:
enc = {"zlib": True, "complevel": 4}
r.to_dataset(name="rain").to_netcdf("../data/processed/rain_daily_box.nc", encoding={"rain": enc})
xr.Dataset({"tmax": tx, "tmin": tn}).to_netcdf("../data/processed/temp_daily_box.nc",
                                              encoding={"tmax": enc, "tmin": enc})